# ChaoSSM — Phase 2d: Hyperparameter Retuning at Scale (§9.2's Prerequisite)

## Why this phase

Phase 2c found the copying advantage collapses to chance by 4x the pilot scale (N=48 -> 192),
with outright training divergence in 2 of 10 chaotic-arm runs at N=192 — something never
observed at pilot scale. Two explanations were left open: a genuine scale limit, or a
hyperparameter artifact, since every setting (`lr`, `gamma_init`) was held fixed across sizes
deliberately to isolate scale as the only variable. This phase is the diagnostic v3.0's §9.2
committed to running before any scaling-phase budget is spent.

## Design

A small factorial sweep at N=96 and N=192, on the copy task, for the two chaotic arms only
(`linear` showed no scale-related problems in Phase 2c and serves as a fixed reference at each
size, not swept):

- **lr**: `3e-4` (pilot default) vs. `1e-4` (lower, in case larger matrices need smaller steps)
- **gamma_init**: `1.0` (pilot default) vs. `0.5` (less aggressive chaos at initialization)

Four configurations (`baseline`, `lower_lr`, `lower_gamma`, `lower_both`) x 2 sizes x 2 chaotic
arms x 4 seeds = 64 runs, plus a `linear` reference at each size (4 seeds) = 72 total. The
`baseline` configuration exactly replicates Phase 2c's setup and serves as an internal
consistency check — if it doesn't reproduce Phase 2c's collapse and divergence, something in
this notebook is wrong and the rest of the sweep shouldn't be trusted.

The corridor's own regularization settings (`corridor_target`, `corridor_half_width`) are held
fixed across all configurations here, to keep this sweep interpretable; retuning those
specifically is noted as further work if this sweep doesn't resolve the question.

**Decision rule, stated before running**: if any configuration recovers accuracy toward the
pilot-scale level (~0.13-0.17) and/or eliminates divergence, the S5.7 collapse is a
hyperparameter artifact and the scaling phase can proceed using that recipe. If none does, the
collapse is genuinely scale-related and S9's task-level claims need to be rescoped before S9.3.

## Setup

In [1]:
import os, json, math, time, warnings, copy as pycopy
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon
from torch.func import jacrev

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def setup_results_dir(project="ChaoSSM", phase="phase2d"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

# Phase 2c's own reported numbers, for direct comparison against this run's "baseline" configuration
PHASE2C_REFERENCE = {
    (96, "linear"): 0.0797, (96, "chaossm_free"): 0.0656, (96, "chaossm_corridor"): 0.1956,
    (192, "linear"): 0.0628, (192, "chaossm_free"): 0.0557, (192, "chaossm_corridor"): 0.0607,
}
PHASE2C_DIVERGENCE = {(192, "chaossm_free"): 1/5, (192, "chaossm_corridor"): 1/5}  # 1 of 5 seeds each
print("Phase 2c reference (baseline hyperparameters):", PHASE2C_REFERENCE)

torch 2.11.0+cu128 | device: cuda
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase2d
Phase 2c reference (baseline hyperparameters): {(96, 'linear'): 0.0797, (96, 'chaossm_free'): 0.0656, (96, 'chaossm_corridor'): 0.1956, (192, 'linear'): 0.0628, (192, 'chaossm_free'): 0.0557, (192, 'chaossm_corridor'): 0.0607}


## Configuration

In [2]:
SMOKE_TEST = False   # full run: 4 hp-configs x 2 sizes x 2 chaotic arms x 4 seeds, + linear reference

@dataclass
class Config:
    state_dim: int = 96   # overwritten by the sweep
    d_model: int = 96
    vocab: int = 16
    steps: int = 1200
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64
    grad_clip: float = 1.0
    gamma_init: float = 1.0
    warmup_frac: float = 0.10
    dual_lr: float = 0.05
    alpha_max: float = 50.0
    corridor_target: float = 0.15
    corridor_half_width: float = 0.10
    lyap_window: int = 4
    lyap_subsample: int = 3
    lyap_every: int = 20
    copy_len: int = 12
    sizes: tuple = (96, 192)
    chaotic_arms: tuple = ("chaossm_free", "chaossm_corridor")
    hp_configs: tuple = (
        ("baseline",    3e-4, 1.0),
        ("lower_lr",    1e-4, 1.0),
        ("lower_gamma", 3e-4, 0.5),
        ("lower_both",  1e-4, 0.5),
    )
    seeds: int = 4

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 150; CFG.seeds = 2
    CFG.hp_configs = (("baseline", 3e-4, 1.0), ("lower_lr", 1e-4, 1.0))
    print("*** SMOKE TEST — pipeline check only ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

{
  "state_dim": "96",
  "d_model": "96",
  "vocab": "16",
  "steps": "1200",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "gamma_init": "1.0",
  "warmup_frac": "0.1",
  "dual_lr": "0.05",
  "alpha_max": "50.0",
  "corridor_target": "0.15",
  "corridor_half_width": "0.1",
  "lyap_window": "4",
  "lyap_subsample": "3",
  "lyap_every": "20",
  "copy_len": "12",
  "sizes": "(96, 192)",
  "chaotic_arms": "('chaossm_free', 'chaossm_corridor')",
  "hp_configs": "(('baseline', 0.0003, 1.0), ('lower_lr', 0.0001, 1.0), ('lower_gamma', 0.0003, 0.5), ('lower_both', 0.0001, 0.5))",
  "seeds": "4"
}


## 1. Model — unchanged, including the dtype fix from Phase 2b/2c

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)

class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)

class ChaoticMap(nn.Module):
    def __init__(self, N, gamma_init=1.0):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return 4.0 * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)

class ChaoSSMCell(nn.Module):
    def __init__(self, cfg, phi_kind):
        super().__init__()
        self.N = cfg.state_dim
        self.phi_kind = phi_kind
        self.phi = LinearMap(cfg.state_dim) if phi_kind == "linear" \
                   else ChaoticMap(cfg.state_dim, cfg.gamma_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.phi_kind != "linear" else torch.tanh(raw)
    def init_state(self, b, dev):
        dtype = self.A.dtype
        return torch.zeros(b, self.N, device=dev, dtype=dtype) if self.phi_kind == "linear" \
               else torch.full((b, self.N), 0.5, device=dev, dtype=dtype)

class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, phi_kind, vocab_size, readout_size):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.cell = ChaoSSMCell(cfg, phi_kind)
        self.readout = nn.Linear(cfg.state_dim, readout_size)
    def forward(self, tok, return_traj=False):
        B, T = tok.shape
        x = self.emb(tok); h = self.cell.init_state(B, tok.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t]); outs.append(self.readout(h))
            if return_traj: traj.append(h)
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        lg = torch.stack(outs, 1)
        return (lg, torch.stack(traj, 1), x) if return_traj else lg

## 2. Estimator, task, accuracy — unchanged

In [4]:
def _full_jac_fn(model, x):
    return lambda z: model.cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]

def lambda_window(model, traj, xseq, W, n_sample):
    B, T, N = traj.shape
    if T <= W + 1:
        return torch.zeros((), device=traj.device)
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_full_jac_fn(model, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()

def make_copy_batch(cfg, batch, device, gen):
    L, V = cfg.copy_len, cfg.vocab
    sep, pad = V, V + 1
    src = torch.randint(0, V, (batch, L), device=device, generator=gen)
    inp = torch.cat([src, torch.full((batch,1), sep, device=device),
                     torch.full((batch,L), pad, device=device)], 1)
    tgt = torch.cat([torch.full((batch,L+1), -100, device=device), src], 1)
    return inp, tgt

VOCAB_SIZE = CFG.vocab + 2
CHANCE = 1.0 / CFG.vocab

def copy_accuracy(logits, tgt):
    mask = tgt != -100
    pred = logits.argmax(-1)
    return ((pred == tgt) & mask).float().sum().item() / mask.float().sum().clamp(min=1).item()

## 3. Trainer — `lr` and `gamma_init` now vary per configuration

In [5]:
def train_one(arm, size, lr, gamma_init, seed, cfg):
    cfg = pycopy.deepcopy(cfg)
    cfg.state_dim = size; cfg.d_model = size; cfg.lr = lr; cfg.gamma_init = gamma_init
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    phi_kind = "linear" if arm == "linear" else "chaossm"
    model = ChaoSSMLM(cfg, phi_kind, VOCAB_SIZE, VOCAB_SIZE).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))

    use_lyap = (arm == "chaossm_corridor")
    lo = hi = None
    if use_lyap:
        lo = cfg.corridor_target - cfg.corridor_half_width
        hi = cfg.corridor_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    warmup_steps = int(cfg.warmup_frac * cfg.steps)
    diverged = False
    final_acc = float("nan"); lam_val = float("nan")

    for step in range(cfg.steps):
        inp, tgt = make_copy_batch(cfg, cfg.batch, DEVICE, gen)
        need_traj = (arm != "linear")
        out = model(inp, return_traj=need_traj)
        logits, traj, xseq = out if need_traj else (out, None, None)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)

        L_pen = torch.zeros((), device=DEVICE)
        if arm != "linear":
            in_warmup = step < warmup_steps
            lam = lambda_window(model, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)
            lam_val = float(lam.detach())
            if use_lyap and not in_warmup:
                L_pen = a_hi * F.relu(lam - hi) + a_lo * F.relu(lo - lam)

        opt.zero_grad(set_to_none=True); (L_task + L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()

        if use_lyap and step >= warmup_steps:
            a_hi = float(np.clip(a_hi + cfg.dual_lr * (lam_val - hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr * (lo - lam_val), 0.0, cfg.alpha_max))

        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break

        if step % 20 == 0:
            with torch.no_grad():
                final_acc = copy_accuracy(logits, tgt)

    return dict(arm=arm, state_dim=size, hp_lr=lr, hp_gamma_init=gamma_init, seed=seed,
               diverged=diverged, final_acc=final_acc, lam_settled=lam_val)

## 4. Run — the sweep

In [6]:
rows = []
total = (len(CFG.hp_configs) * len(CFG.sizes) * len(CFG.chaotic_arms) * CFG.seeds
        + len(CFG.sizes) * CFG.seeds)
done = 0

for size in CFG.sizes:
    for hp_name, lr, gamma_init in CFG.hp_configs:
        for arm in CFG.chaotic_arms:
            for sd in range(CFG.seeds):
                r = train_one(arm, size, lr, gamma_init, sd, CFG)
                r["hp_config"] = hp_name
                rows.append(r); done += 1
                flag = " DIVERGED" if r["diverged"] else ""
                print(f"[{done}/{total}] N={size:4d} {hp_name:12s} {arm:18s} seed={sd}  "
                      f"final_acc={r['final_acc']:.4f} (chance {CHANCE:.4f})  "
                      f"lam={r['lam_settled']:+.3f}{flag}")

for size in CFG.sizes:
    for sd in range(CFG.seeds):
        r = train_one("linear", size, 3e-4, 1.0, sd, CFG)
        r["hp_config"] = "reference"
        rows.append(r); done += 1
        print(f"[{done}/{total}] N={size:4d} {'reference':12s} {'linear':18s} seed={sd}  "
              f"final_acc={r['final_acc']:.4f} (chance {CHANCE:.4f})")

SWEEP = pd.DataFrame(rows)
savetable(SWEEP, "phase2d_sweep.csv")

[1/72] N=  96 baseline     chaossm_free       seed=0  final_acc=0.0482 (chance 0.0625)  lam=+0.884
[2/72] N=  96 baseline     chaossm_free       seed=1  final_acc=0.0716 (chance 0.0625)  lam=+1.333
[3/72] N=  96 baseline     chaossm_free       seed=2  final_acc=0.0677 (chance 0.0625)  lam=+1.248
[4/72] N=  96 baseline     chaossm_free       seed=3  final_acc=0.0638 (chance 0.0625)  lam=+0.973
[5/72] N=  96 baseline     chaossm_corridor   seed=0  final_acc=0.2122 (chance 0.0625)  lam=+0.239
[6/72] N=  96 baseline     chaossm_corridor   seed=1  final_acc=0.2214 (chance 0.0625)  lam=+0.198
[7/72] N=  96 baseline     chaossm_corridor   seed=2  final_acc=0.2344 (chance 0.0625)  lam=+0.222
[8/72] N=  96 baseline     chaossm_corridor   seed=3  final_acc=0.0651 (chance 0.0625)  lam=+0.386
[9/72] N=  96 lower_lr     chaossm_free       seed=0  final_acc=0.1367 (chance 0.0625)  lam=+0.417
[10/72] N=  96 lower_lr     chaossm_free       seed=1  final_acc=0.1680 (chance 0.0625)  lam=+0.278
[11/72] N

## 5. Internal consistency check — does `baseline` replicate Phase 2c?

Before trusting any comparison across configurations, the `baseline` configuration (identical
settings to Phase 2c) should reproduce Phase 2c's own numbers reasonably closely. If it
doesn't, something in this notebook's setup differs from Phase 2c's and the rest of the sweep
should not be trusted without first finding that discrepancy.

In [7]:
check_rows = []
for size in CFG.sizes:
    for arm in CFG.chaotic_arms + ("linear",):
        hp = "baseline" if arm != "linear" else "reference"
        d = SWEEP[(SWEEP.state_dim==size) & (SWEEP.arm==arm) & (SWEEP.hp_config==hp)]
        this_run = d.final_acc.mean()
        this_div = d.diverged.mean()
        ref = PHASE2C_REFERENCE.get((size, arm), float("nan"))
        ref_div = PHASE2C_DIVERGENCE.get((size, arm), 0.0)
        check_rows.append(dict(state_dim=size, arm=arm, this_run_acc=float(this_run),
                               phase2c_acc=float(ref), this_run_divergence=float(this_div),
                               phase2c_divergence=float(ref_div)))
CHECK = pd.DataFrame(check_rows)
print(CHECK.round(4).to_string(index=False))
savetable(CHECK, "phase2d_consistency_check.csv")

max_gap = (CHECK.this_run_acc - CHECK.phase2c_acc).abs().max()
print(f"\nmax |this_run - phase2c| accuracy gap: {max_gap:.4f}")
print("CONSISTENCY CHECK:", "PASS" if max_gap < 0.05 else "WARNING -- investigate before trusting the sweep")

 state_dim              arm  this_run_acc  phase2c_acc  this_run_divergence  phase2c_divergence
        96     chaossm_free        0.0628       0.0656                 0.00                 0.0
        96 chaossm_corridor        0.1833       0.1956                 0.00                 0.0
        96           linear        0.0801       0.0797                 0.00                 0.0
       192     chaossm_free        0.0537       0.0557                 0.00                 0.2
       192 chaossm_corridor        0.0618       0.0607                 0.25                 0.2
       192           linear        0.0625       0.0628                 0.00                 0.0
saved /content/drive/My Drive/ChaoSSM/results/phase2d/phase2d_consistency_check.csv

max |this_run - phase2c| accuracy gap: 0.0123
CONSISTENCY CHECK: PASS


## 6. Does any configuration recover the pilot-scale advantage or eliminate divergence?

In [8]:
summary = SWEEP[SWEEP.hp_config != "reference"].groupby(
    ["state_dim","hp_config","arm"])[["final_acc","diverged"]].agg(
    final_acc_mean=("final_acc","mean"), final_acc_std=("final_acc","std"),
    divergence_rate=("diverged","mean")).round(4)
print(summary.to_string())
savetable(summary.reset_index(), "phase2d_summary.csv")

def paired(size, hp, arm, col="final_acc"):
    x = SWEEP[(SWEEP.state_dim==size)&(SWEEP.hp_config==hp)&(SWEEP.arm==arm)].sort_values("seed")[col].values
    y = SWEEP[(SWEEP.state_dim==size)&(SWEEP.hp_config=="baseline")&(SWEEP.arm==arm)].sort_values("seed")[col].values
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(state_dim=size, hp_config=hp, arm=arm, n=len(x), p=np.nan,
                    median_diff=float(np.median(x-y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(state_dim=size, hp_config=hp, arm=arm, n=len(x), p=float(p), median_diff=float(np.median(x-y)))

stat_rows = []
for size in CFG.sizes:
    for hp_name, _, _ in CFG.hp_configs:
        if hp_name == "baseline": continue
        for arm in CFG.chaotic_arms:
            stat_rows.append(paired(size, hp_name, arm))
ST = pd.DataFrame(stat_rows)
print()
print(ST.to_string(index=False))
savetable(ST, "phase2d_stats.csv")
print(f"\nmin achievable two-sided p at n={CFG.seeds}: {2/(2**CFG.seeds):.4f}")

PILOT_SCALE_LEVEL = 0.13  # approximate N=48 chaossm_free/corridor accuracy from Phase 2c
any_recovered = bool((summary["final_acc_mean"] > PILOT_SCALE_LEVEL * 0.8).any())
any_divergence_eliminated = bool(
    (summary.reset_index().query("hp_config != 'baseline'").divergence_rate == 0).any() and
    (summary.reset_index().query("hp_config == 'baseline'").divergence_rate > 0).any()
)
print(f"\nany configuration recovers >=80% of pilot-scale accuracy: {any_recovered}")
print(f"any configuration eliminates divergence seen at baseline: {any_divergence_eliminated}")

                                        final_acc_mean  final_acc_std  divergence_rate
state_dim hp_config   arm                                                             
96        baseline    chaossm_corridor          0.1833         0.0793             0.00
                      chaossm_free              0.0628         0.0103             0.00
          lower_both  chaossm_corridor          0.1794         0.0050             0.00
                      chaossm_free              0.1784         0.0061             0.00
          lower_gamma chaossm_corridor          0.2490         0.0412             0.00
                      chaossm_free              0.2598         0.0424             0.00
          lower_lr    chaossm_corridor          0.1579         0.0128             0.00
                      chaossm_free              0.1387         0.0216             0.00
192       baseline    chaossm_corridor          0.0618         0.0095             0.25
                      chaossm_free         

In [9]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
markers = {"chaossm_free":"o", "chaossm_corridor":"s"}
for i, size in enumerate(CFG.sizes):
    d = summary.reset_index().query("state_dim == @size")
    for arm in CFG.chaotic_arms:
        dd = d[d.arm == arm]
        ax[i].bar([f"{h}\n{arm.split('_')[1]}" for h in dd.hp_config], dd.final_acc_mean,
                 yerr=dd.final_acc_std, alpha=.8, label=arm if i==0 else None)
    ax[i].axhline(CHANCE, color="r", ls="--", label="chance")
    ax[i].axhline(PILOT_SCALE_LEVEL, color="g", ls=":", label="pilot-scale (N=48) level")
    ax[i].set(title=f"N={size}", ylabel="accuracy" if i==0 else "")
    ax[i].tick_params(axis="x", labelsize=7)
    ax[i].legend(fontsize=7)
fig.tight_layout(); savefig(fig, "phase2d_recovery.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase2d/phase2d_recovery.png


## 7. Verdict

| Observation | Reading |
|---|---|
| Consistency check fails (large gap vs. Phase 2c) | Stop — something differs from Phase 2c's setup in this notebook. Find and fix before trusting any other result here. |
| Some configuration recovers >=80% of pilot-scale accuracy and/or eliminates divergence | S5.7's collapse was a hyperparameter artifact. Adopt that configuration for S9.3, and note per-scale tuning as a standard requirement in the manuscript rather than a ChaoSSM-specific weakness. |
| No configuration meaningfully improves on baseline | S5.7's collapse is genuinely scale-related within the range tested. Per the proposal's own decision rule, S9's task-level claims need to be explicitly rescoped before committing to S9.3 — report the copying win as pilot-scale-specific rather than a general claim. |
| Divergence persists regardless of configuration | The chaotic map's stability at scale needs an architectural fix (e.g., a numerically better-conditioned parameterization of the affine part in §4.3), not just a training-recipe fix — a finding for a future revision of §4.3, not this diagnostic alone. |

In [10]:
verdict = dict(
    smoke_test=SMOKE_TEST,
    consistency_check=CHECK.to_dict("records"),
    consistency_max_gap=float(max_gap),
    sweep_summary=summary.reset_index().to_dict("records"),
    stats=ST.to_dict("records"),
    any_recovered_80pct_pilot_level=any_recovered,
    any_divergence_eliminated=any_divergence_eliminated,
    decision="hyperparameter artifact -- adopt retuned recipe for S9.3" if any_recovered else
             "genuine scale limit within tested range -- rescope S9 task-level claims before S9.3",
    config=dict(steps=CFG.steps, seeds=CFG.seeds, sizes=CFG.sizes,
                hp_configs=[c[0] for c in CFG.hp_configs]),
)
savejson(verdict, "phase2d_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase2d/phase2d_verdict.json
{
  "smoke_test": false,
  "consistency_check": [
    {
      "state_dim": 96,
      "arm": "chaossm_free",
      "this_run_acc": 0.06282552083333333,
      "phase2c_acc": 0.0656,
      "this_run_divergence": 0.0,
      "phase2c_divergence": 0.0
    },
    {
      "state_dim": 96,
      "arm": "chaossm_corridor",
      "this_run_acc": 0.18326822916666666,
      "phase2c_acc": 0.1956,
      "this_run_divergence": 0.0,
      "phase2c_divergence": 0.0
    },
    {
      "state_dim": 96,
      "arm": "linear",
      "this_run_acc": 0.080078125,
      "phase2c_acc": 0.0797,
      "this_run_divergence": 0.0,
      "phase2c_divergence": 0.0
    },
    {
      "state_dim": 192,
      "arm": "chaossm_free",
      "this_run_acc": 0.0537109375,
      "phase2c_acc": 0.0557,
      "this_run_divergence": 0.0,
      "phase2c_divergence": 0.2
    },
    {
      "state_dim": 192,
      "arm": "chaossm_corridor",
      "this_run_